# 10 — Criterios de la puerta P4 sobre los modelos ajustados, y puerta P3 para todo conjunto donde se ajusta un modelo

**Lugar en el framework.** Etapa E9 (sensibilidad y robustez) y puerta P4 (criterios C1 y C2), con la puerta P3 aplicada a todo conjunto en el que se ajusta un modelo. Versión en inglés: `10_g4_criteria_fitted_models.ipynb` (mismo código).

**Regla que se aplica.** Un criterio de la puerta P4 que pregunta cómo se comporta un modelo en una parte de su población (C1: población *landmark*; C2: desenlace incidente) se evalúa sobre **el modelo ajustado una sola vez en toda su población**: sus predicciones fuera de muestra se restringen al subconjunto y no se vuelve a ajustar allí. Un modelo solo se ajusta en un conjunto que pase la puerta P3; un criterio que necesitaría ajustar en un conjunto que no pasa P3 se reporta como **no evaluable**.

| Sección | Framework | Qué produce |
|---|---|---|
| 1 | E1 · E4 / P1 | Configuración, datos, marcas de tiempo y conteos de referencia (como el notebook 08) |
| 2 | E6 · E7 | Especificaciones de los cuatro modelos: principal, retorno R4 (laboratorios ampliados), retorno R2 (escalamiento) y retorno R2 tras la reducción por tamaño de muestra |
| 3 | E8 | Validación cruzada anidada agrupada por paciente de cada modelo (una vez) |
| 4 | E9 · P4 (C1, C2) | C1 y C2 sobre las predicciones ajustadas, intervalos de 95% por *bootstrap* de pacientes para toda cantidad de decisión, componentes del escalamiento y su solapamiento |
| 5 | P3 | Puerta P3 para los períodos de desarrollo de la validación temporal (C4) y para las poblaciones incidentes |

Reportes (solo agregados): `data/outputs_g4_fitted_models.txt` y `data/outputs_g3_fitted_subsets.txt`. Requiere las tablas del notebook 08, incluida `mimiciv.escalation_v3`. Tiempo: cerca de 1 hora.

### 1. Librerías, conexión y configuración

> **Framework:** E1. Misma semilla, particiones, repeticiones y grillas que los notebooks 02 y 08.

In [ ]:
import os, re, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

### Datos y diccionario

> **Framework:** E7 y E4 / P1 (verificación contra los conteos de referencia congelados).

In [ ]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

### Marcas de tiempo

> **Framework:** E6 y E4. Clase temporal de cada evento de SMM respecto al punto de predicción T.

In [ ]:
# Timing classes (one row per eligible delivery hospitalization)
tf = pd.read_sql("""SELECT hadm_id, smm_timing_class, smm_incident_strict, aki_prevalent, aki_incident
                     FROM mimiciv.timing_flags_v3""", engine)
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    tf[col] = tf[col].fillna(False).astype(bool)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    df[col] = df[col].fillna(False).astype(bool)
df["y_aki"] = df["aki_incident"].astype(int)
n_tf = int((df.eligible_24h & df.is_delivery_v2 & (df.smm_timing_class != "not an eligible delivery")).sum())
print(f"Eligible deliveries with timing flags: {n_tf} (expected {EXPECTED['eligible_deliveries']}) "
      f"[{'OK' if n_tf == EXPECTED['eligible_deliveries'] else 'CHECK'}]")

### 2. Poblaciones y etiquetas

> **Framework:** E6. Mismas especificaciones del notebook 08.

In [ ]:
def make_spec(name):
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    prevalent_or_indet = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])
    if name == "main":
        pass
    elif name in ("landmark", "incident_documented", "incident_strict"):
        d = d[~d.icu_in_24h & ~prevalent_or_indet]
        if name == "incident_documented":
            d = d[~(d.y.astype(bool) & (d.smm_timing_class == "4 untimed"))]
        if name == "incident_strict":
            d = d[~(d.y.astype(bool) & ~d.smm_incident_strict)]
    elif name == "aki_lab_incident":
        d = d[~d.icu_in_24h & ~d.aki_prevalent]
        label = "y_aki"
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

for nm in ["main", "landmark", "incident_documented", "incident_strict"]:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"prevalence={d[sp['label']].mean():.4f}")

### Funciones base (preprocesamiento, algoritmos, ajuste anidado, ensamble, calibración)

> **Framework:** E8. Copiadas del notebook 08 sin cambios.

In [ ]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=""):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

### Métricas y *bootstrap* de pacientes

> **Framework:** E8. Copiadas del notebook 08.

In [ ]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

### Funciones de reporte

> **Framework:** E11. Copiadas del notebook 08 (`emit` guarda reportes agregados en `data/`).

In [ ]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label="ideal")
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label="Logistic regression")
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label="Stacked ensemble")
    ax.set_xlabel("Mean predicted risk (decile)"); ax.set_ylabel("Observed SMM rate")
    ax.set_title(f"Calibration by decile -- {res['spec']['name']}"); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Print the aggregate report and also keep it as a text file (aggregates only) so it can be shared without truncation."""
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    print(text)
    path = f"../data/outputs_{name}.txt"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    print(sl[sl.repeat == 1].drop(columns=["repeat"]).to_string(index=False))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    print(pd.DataFrame(rows).to_string(index=False))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    print(pd.DataFrame(rows).to_string(index=False))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        print(pd.DataFrame(rows).to_string(index=False))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    print(decile_table(res).to_string(index=False))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    print(tab.to_string(index=False))

### Escalamiento de la atención después de T (retorno R2)

> **Framework:** Retorno R2, etiqueta de `sql/07_timing_audit/04_escalation_label.sql`.

In [ ]:
EXPECTED_ESC = {"population": 19268, "escalation": 101}   # Q-E1 of sql/07_timing_audit/03_escalation_counts.sql
esc = pd.read_sql("""SELECT hadm_id, icu_after, vent_after, smm_proc_after, death_after, escalation_any
                     FROM mimiciv.escalation_v3""", engine)
for c in ["icu_after", "vent_after", "smm_proc_after", "death_after", "escalation_any"]:
    esc[c] = esc[c].fillna(False).astype(bool)
chk = {"population": len(esc), "escalation": int(esc.escalation_any.sum())}
for k, v in chk.items():
    print(f"escalation {k}: {v}  (expected {EXPECTED_ESC[k]})  [{'OK' if v == EXPECTED_ESC[k] else 'CHECK: differs from Q-E1'}]")

df = df.drop(columns=[c for c in esc.columns if c != "hadm_id" and c in df.columns])
df = df.merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

if "_make_spec_s10b" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10b = make_spec
ESC11 = ["escalation", "escalation_labs", "escalation_landmark"]

def make_spec(name):
    if name not in ESC11:
        return _make_spec_s10b(name)
    sp = _make_spec_s10b("main")
    d = sp["data"]
    d = d[d.in_esc_pop & ~d.icu_in_24h]
    if name == "escalation_landmark":
        d = d[~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    d = d.reset_index(drop=True)
    num, binc = list(sp["num"]), list(sp["bin"])
    if name == "escalation_labs":
        num = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label="y_esc")

### Funciones de la puerta P3 (Riley et al.)

> **Framework:** P3. Copiadas del notebook 08.

In [ ]:
def riley(P, phi, r2cs, shrink=0.9, deltas=(0.05, 0.01)):
    lnl0 = phi * math.log(phi) + (1 - phi) * math.log(1 - phi)
    max_r2 = 1 - math.exp(2 * lnl0)
    n1 = P / ((shrink - 1) * math.log(1 - r2cs / shrink))
    s_vh = r2cs / (r2cs + 0.05 * max_r2)
    n2 = P / ((s_vh - 1) * math.log(1 - r2cs / s_vh))
    out = {"max_r2cs": max_r2, "n_shrinkage": n1, "n_optimism": n2}
    for dl in deltas:
        out[f"n_risk_delta{dl}"] = (1.96 / dl) ** 2 * phi * (1 - phi)
    n = max(n1, n2, out[f"n_risk_delta{deltas[-1]}"])
    out["n_required"] = n
    out["events_required"] = n * phi
    out["epp_at_required"] = n * phi / P
    out["P_max_supported"] = None
    return out

def p_max(n, r2cs, shrink=0.9):
    """Largest number of parameters with expected shrinkage >= 0.9 for a given n and R2 (criterion 1 solved for P)."""
    return n * (shrink - 1) * math.log(1 - r2cs / shrink)

def riley_block(label, n, events, P, r2_list):
    phi = events / n
    print(f"\n{label}: n={n}, events={events}, prevalence={phi:.5f}, candidate parameters P={P}, EPP={events/P:.2f}")
    rows = []
    for tag, r2 in r2_list:
        rr = riley(P, phi, r2)
        rows.append({"R2_CS scenario": tag, "R2_CS": f"{r2:.4f}", "max R2_CS": f"{rr['max_r2cs']:.4f}",
                     "n shrinkage": f"{rr['n_shrinkage']:.0f}", "n optimism": f"{rr['n_optimism']:.0f}",
                     "n risk(+-0.05)": f"{rr['n_risk_delta0.05']:.0f}", "n risk(+-0.01)": f"{rr['n_risk_delta0.01']:.0f}",
                     "n required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                     "available n": n, "available events": events,
                     "max P supported (n,R2)": f"{p_max(n, r2):.1f}"})
    print(pd.DataFrame(rows).to_string(index=False))

def count_parameters(spec):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    return int(make_prep(spec["num"], spec["bin"], spec["cat"]).fit(X).transform(X).shape[1])

def r2cs_from_oof(res):
    y = res["y"].astype(float); n = len(y); phi = y.mean()
    lnl0 = n * (phi * math.log(phi) + (1 - phi) * math.log(1 - phi))
    vals = []
    for r in range(N_REPEATS):
        p = np.clip(res["preds"]["LR"][r], 1e-9, 1 - 1e-9)
        lnl1 = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
        vals.append(max(1 - math.exp(2 / n * (lnl0 - lnl1)), 1e-6))
    return float(np.mean(vals))

### Conjunto reducido de predictores del escalamiento

> **Framework:** reducción en P3, conjunto registrado en `docs/decision_G3_reduction_2026-11-09.md`.

In [ ]:
def add_reduced_columns(frame):
    """Columns of the reduced set fixed in (docs/decision_G3_reduction_2026-11-09.md). Row-wise, no outcome information."""
    eth = frame["ethnicity"].fillna("").astype(str).str.upper()
    frame["race5"] = np.select(
        [eth.str.startswith("WHITE"), eth.str.startswith("BLACK"),
         eth.str.startswith("HISPANIC") | eth.str.startswith("SOUTH AMERICAN"), eth.str.startswith("ASIAN")],
        ["White", "Black", "Hispanic or Latina", "Asian"], default="Other or unknown")
    frame["ins_medicaid"] = (frame["insurance"].astype(str) == "Medicaid").astype(int)
    frame["lang_english"] = (frame["language"].astype(str) == "English").astype(int)
    frame["adm_urgent"] = (frame["admission_type"].astype(str) == "URGENT").astype(int)
    frame["loc_physician"] = (frame["admission_location"].astype(str) == "PHYSICIAN REFERRAL").astype(int)
    return frame

RED_BIN = ["ins_medicaid", "lang_english", "adm_urgent", "loc_physician"]
RED_CAT = ["race5"]
P_MAX_REDUCTION = 16

df = add_reduced_columns(df)
if "_make_spec_s15" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s15 = make_spec
RED15 = ["escalation_reduced", "escalation_landmark_reduced"]

def make_spec(name):
    if name not in RED15:
        return _make_spec_s15(name)
    sp = _make_spec_s15(name.replace("_reduced", ""))
    d = add_reduced_columns(sp["data"].copy())
    binc = [b for b in (["creatinine_missing"] + RED_BIN) if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=list(sp["num"]), bin=binc, cat=list(RED_CAT), label="y_esc")

### Especificaciones de los cuatro modelos

In [ ]:
# Specification of the model of return R4 in the full population (main predictors + every laboratory test of the
# dictionary with role "laboratory sensitivity", each with its missingness indicator), as in notebooks 02 and 08.
if "_make_spec_s10r" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10r = make_spec

def make_spec(name):
    if name != "labs_extended":
        return _make_spec_s10r(name)
    sp = _make_spec_s10r("main")
    d = sp["data"]
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

FITTED = ["main", "labs_extended", "escalation", "escalation_reduced"]
for nm in FITTED:
    sp = make_spec(nm)
    print(f"{nm:20s} admissions={len(sp['data']):6d}  events={int(sp['data'][sp['label']].sum()):4d}  "
          f"parameters={count_parameters(sp)}")

### 3. Ajustar cada modelo una vez

> **Framework:** E8.

In [ ]:
# One patient-grouped nested cross-validation per model (the same procedure as notebooks 02 and 08). Each admission
# receives exactly one out-of-fold prediction per repeat; these predictions are reused in every subset below, so no
# model is refitted inside a subset. About 10-15 minutes per model.
fitted = {}
for nm in FITTED:
    fitted[nm] = run_cv(make_spec(nm))

### 4. Subconjuntos de cada población ajustada

> **Framework:** E9 · P4. Población *landmark* (C1), SMM incidente documentada y estricta (C2) y, para el escalamiento, cada componente frente a las admisiones sin escalamiento.

In [ ]:
# Subsets of each fitted population, as boolean masks over its rows, with the label used in each subset.
def smm_subsets(d):
    prev_or_ind = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"]).values
    icu = d.icu_in_24h.values.astype(bool)
    y = d.y.values.astype(bool)
    landmark = ~icu & ~prev_or_ind
    untimed = (d.smm_timing_class == "4 untimed").values
    strict = d.smm_incident_strict.values.astype(bool)
    return {
        "full population": (np.ones(len(d), bool), y),
        "landmark (C1)": (landmark, y),
        "documented incident SMM (C2)": (landmark & ~(y & untimed), y),
        "strict incident SMM (C2)": (landmark & ~(y & ~strict), y),
    }

def escalation_subsets(d):
    prev_or_ind = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"]).values
    y = d.y_esc.values.astype(bool)
    out = {"full population": (np.ones(len(d), bool), y),
           "landmark (C1)": (~prev_or_ind, y)}
    for comp, lab in (("icu_after", "ICU admission"), ("vent_after", "mechanical ventilation"),
                      ("smm_proc_after", "procedure-based SMM indicator"), ("death_after", "in-hospital death")):
        c = d[comp].fillna(False).values.astype(bool) & y
        out[f"component: {lab} vs no escalation"] = ((~y) | c, c)
    return out

SUBSETS = {"main": smm_subsets, "labs_extended": smm_subsets,
           "escalation": escalation_subsets, "escalation_reduced": escalation_subsets}
EVAL_MODELS = ["LR", "XGB", "Stack"]
MIN_EVENTS = 10   # a subset with fewer events is reported with counts only

### Cantidades de decisión con intervalos de 95%

> **Framework:** P4. Se remuestrean los mismos pacientes en todos los subconjuntos, así que la caída de C1 y la ganancia de AUPRC conservada tienen intervalos pareados.

In [ ]:
# Metrics on the already fitted out-of-fold predictions, averaged over repeats, with patient-level bootstrap intervals
# (the same resampling of patients for every subset, so differences between subsets are paired).
B_DEC = 50 if QUICK else 1000

def subset_metrics(y_all, preds, w, mask, lab, models):
    out = {}
    m = mask & (w > 0)
    yy, ww = lab[m].astype(float), w[m]
    if np.sum(ww * yy) < 1 or np.sum(ww * (1 - yy)) < 1:
        return None
    prev = np.sum(ww * yy) / ww.sum()
    for mod in models:
        a, p_ = [], []
        for r in range(N_REPEATS):
            pp = preds[mod][r][m]
            a.append(roc_auc_score(yy, pp, sample_weight=ww))
            p_.append(average_precision_score(yy, pp, sample_weight=ww))
        out[(mod, "auroc")] = float(np.mean(a)); out[(mod, "auprc")] = float(np.mean(p_))
        out[(mod, "gain")] = out[(mod, "auprc")] / prev
    return out

def evaluate(nm):
    res = fitted[nm]; d = res["spec"]["data"]; y = res["y"]; g = res["g"]; preds = res["preds"]
    subs = SUBSETS[nm](d)
    w1 = np.ones(len(y))
    point = {s: subset_metrics(y, preds, w1, mk, lab, EVAL_MODELS) for s, (mk, lab) in subs.items()}
    ug, inv = np.unique(g, return_inverse=True); rng = np.random.default_rng(SEED + 11)
    boot = {s: [] for s in subs}
    for b in range(B_DEC):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
        for s, (mk, lab) in subs.items():
            boot[s].append(subset_metrics(y, preds, w, mk, lab, EVAL_MODELS))
    return dict(subs=subs, point=point, boot=boot, d=d)

def ci(vals, d=3):
    v = np.array([x for x in vals if x is not None and np.isfinite(x)])
    if len(v) < 20:
        return ""
    return f" ({np.percentile(v, 2.5):.{d}f}-{np.percentile(v, 97.5):.{d}f})"

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

evals = {}
for nm in FITTED:
    t0 = time.time()
    evals[nm] = evaluate(nm)
    print(f"[{nm}] evaluated in {(time.time() - t0) / 60:.1f} min")

### Reporte de los criterios de P4 sobre los modelos ajustados

In [ ]:
# Report: decision quantities of gate G4 on the fitted models, with 95% intervals for every quantity.
# C1: AUROC drop from the full to the landmark population <= 0.05 AND AUPRC gain kept >= 2/3.
# C2: AUROC >= 0.75 AND lower 95% limit >= 0.70 (documented and strict incident SMM).
def g4_fitted_report():
    print(f"GATE G4 CRITERIA ON THE FITTED MODELS (no refitting inside subsets), QUICK={QUICK}, bootstrap B={B_DEC}")
    print("Each model was fitted once on its whole population (patient-grouped nested cross-validation); every subset uses")
    print("the same out-of-fold predictions. Counts below 10 are shown as <10.")
    for nm in FITTED:
        ev = evals[nm]; d = ev["d"]
        print("\n" + "=" * 110)
        print(f"MODEL {nm}: {len(d)} admissions, {int(fitted[nm]['y'].sum())} events, parameters={count_parameters(fitted[nm]['spec'])}")
        print("=" * 110)
        rows = []
        for s, (mk, lab) in ev["subs"].items():
            nev = int(lab[mk].sum())
            pt = ev["point"][s]
            if pt is None or nev < MIN_EVENTS:
                rows.append({"subset": s, "admissions": _n(mk.sum()), "events": _n(nev), "model": "-", "AUROC": "counts only"})
                continue
            for mod in EVAL_MODELS:
                bt = [b[(mod, "auroc")] if b else np.nan for b in ev["boot"][s]]
                lo = np.nanpercentile(bt, 2.5)
                row = {"subset": s, "admissions": _n(mk.sum()), "events": _n(nev), "model": mod,
                       "AUROC": f"{pt[(mod, 'auroc')]:.3f}" + ci(bt),
                       "AUPRC": f"{pt[(mod, 'auprc')]:.3f}" + ci([b[(mod, 'auprc')] if b else np.nan for b in ev['boot'][s]])}
                if "(C2)" in s or (nm.startswith("escalation") and s == "full population"):
                    row["C2"] = "met" if (pt[(mod, "auroc")] >= 0.75 and lo >= 0.70) else "not met"
                rows.append(row)
        print(pd.DataFrame(rows).fillna("").to_string(index=False))
        # C1 decision quantities with paired intervals
        full, lm = "full population", "landmark (C1)"
        rows = []
        for mod in EVAL_MODELS:
            pf, pl = ev["point"][full], ev["point"][lm]
            drop = pf[(mod, "auroc")] - pl[(mod, "auroc")]
            kept = (pl[(mod, "gain")] - 1) / (pf[(mod, "gain")] - 1) if pf[(mod, "gain")] > 1 else np.nan
            bd = [bf[(mod, "auroc")] - bl[(mod, "auroc")] for bf, bl in zip(ev["boot"][full], ev["boot"][lm]) if bf and bl]
            bk = [(bl[(mod, "gain")] - 1) / (bf[(mod, "gain")] - 1) for bf, bl in zip(ev["boot"][full], ev["boot"][lm])
                  if bf and bl and bf[(mod, "gain")] > 1]
            rows.append({"model": mod, "AUROC full": f"{pf[(mod, 'auroc')]:.3f}", "AUROC landmark": f"{pl[(mod, 'auroc')]:.3f}",
                         "AUROC drop": f"{drop:.3f}" + ci(bd), "AUPRC gain kept": f"{kept:.2f}" + ci(bk, 2),
                         "C1": "met" if (drop <= 0.05 and kept >= 2 / 3) else "not met"})
        print("\nCriterion C1 (drop <= 0.05 and gain kept >= 2/3), paired patient bootstrap")
        print(pd.DataFrame(rows).to_string(index=False))
        if nm.startswith("escalation"):
            e = d[d.y_esc == 1]
            comb = (e.icu_after.fillna(False).astype(int).astype(str) + e.vent_after.fillna(False).astype(int).astype(str)
                    + e.smm_proc_after.fillna(False).astype(int).astype(str) + e.death_after.fillna(False).astype(int).astype(str))
            names = {"1000": "ICU only", "0100": "ventilation only", "0010": "procedure only", "0001": "death only"}
            tab = comb.value_counts().rename_axis("ICU/vent/procedure/death").to_frame("events")
            tab["pattern"] = [names.get(k, "two or more components") for k in tab.index]
            tab = tab.groupby("pattern")["events"].sum()
            print("\nOverlap of the components among escalation events (an event can have several components)")
            print(tab.map(_n).to_string())
            print(f"With an SMM code: {_n(int(e.y.sum()))} of {len(e)}")

emit("g4_fitted_models", g4_fitted_report)

### 5. Puerta P3 para todo conjunto donde se ajusta un modelo

> **Framework:** P3. Períodos de desarrollo de la validación temporal (notebook 09) y poblaciones incidentes.

In [ ]:
# Gate G3 for the subsets in which a model would be FITTED: the development periods of the temporal validation
# (criterion C4, notebook 09) and, for reference, the incident populations that notebook 08 refitted.
# R2_CS: Cox-Snell R2 of the cross-validated logistic regression of the corresponding fitted model above
# (whole period; the development periods are not refitted here). n and events of the development periods are read
# from the aggregate reports of notebook 09.
def dev_counts(fname):
    txt = open(f"../data/{fname}", encoding="utf-8").read()
    m = re.search(r"Development \(year_hi < 2017\): (\d+) admissions, \d+ patients, (\d+) events", txt)
    return int(m.group(1)), int(m.group(2))

R2 = {nm: r2cs_from_oof(fitted[nm]) for nm in FITTED}
PAR = {nm: count_parameters(fitted[nm]["spec"]) for nm in FITTED}

def g3_subsets_report():
    print(f"GATE G3 FOR THE SUBSETS IN WHICH A MODEL IS FITTED (QUICK={QUICK})")
    print("R2_CS observed (cross-validated LR of the whole-period model): " + ", ".join(f"{k} {v:.4f}" for k, v in R2.items()))
    rows = []
    temporal = [("main", "outputs_cut2017.txt", "C4, main model"),
                ("main", "outputs_cut2017_landmark.txt", "C4, main model, landmark"),
                ("labs_extended", "outputs_cut2017_landmark_labs.txt", "C4, return R4, landmark"),
                ("escalation", "outputs_cut2017_escalation.txt", "C4, return R2"),
                ("escalation_reduced", "outputs_cut2017_escalation_reduced.txt", "C4, return R2 after reduction")]
    for nm, f, what in temporal:
        n, ev = dev_counts(f)
        rr = riley(PAR[nm], ev / n, R2[nm])
        rows.append({"fitted on": f"development period before 2017 ({what})", "parameters": PAR[nm], "admissions": n,
                     "events": ev, "admissions required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                     "G3": "passes" if rr["n_required"] <= n else "does NOT pass"})
    for nm in ("main", "labs_extended"):
        d = fitted[nm]["spec"]["data"]
        for s in ("documented incident SMM (C2)", "strict incident SMM (C2)"):
            mk, lab = smm_subsets(d)[s]
            n, ev = int(mk.sum()), int(lab[mk].sum())
            rr = riley(PAR[nm], ev / n, R2[nm])
            rows.append({"fitted on": f"{s}, if refitted ({nm})", "parameters": PAR[nm], "admissions": n, "events": ev,
                         "admissions required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                         "G3": "passes" if rr["n_required"] <= n else "does NOT pass"})
    print(pd.DataFrame(rows).to_string(index=False))
    print("A model whose fitting set does not pass G3 is not evaluated for the criterion that needs it (reported as not assessable).")

emit("g3_fitted_subsets", g3_subsets_report)